In [ ]:
using LensFactory
using LensFactory.Constants
using CairoMakie

In [ ]:
# Initialize default cosmology
cosmo = Cosmology.init_cosmology()

# Lens and source redshifts
zl = 0.5
zs = 1.5

# ADDs and distance ratio
Dol = Cosmology.angular_diameter_distance(cosmo, 0., zl)
Dls = Cosmology.angular_diameter_distance(cosmo, zl, zs)
Dos = Cosmology.angular_diameter_distance(cosmo, 0., zs)
adis = Dls / Dos

# Create image plane grid (default units are ANGLE_ARCSEC)
x, y = Lenses.get_meshgrid(5, 5, 0.01 + 1.0E-6);

In [ ]:
# Create lens
lens = Lenses.init_PIEPLens(v_d=300, x_s=0.5, eps=0.2, pa=30.0)
dxx, dyy, dxy = Lenses.get_jacobian(lens, x, y)

# Generate singularity map
ra31, ra32, rd4 = SingularityMap.from_jacobian(dxx, dyy, dxy, x, y; adis=0.5, D4=true)
fig, ax = Lenses.plot_image_plane(lens, x, y, 0.5; plot_caustic=false)

scatter!(ax, ra31[1, :], ra31[2, :], markersize=15, color=:black)
scatter!(ax, ra32[1, :], ra32[2, :], markersize=10, color=:green)
scatter!(ax, rd4[1, :], rd4[2, :], markersize=20, color=:red)

display(fig)

In [ ]:
# Create image plane grid (default units are ANGLE_ARCSEC)
x, y = Lenses.get_meshgrid(10, 10, 0.05 + 1.0E-6);

# Create a complex lens with n-components
n = 2
comp::Vector{NamedTuple} = []
for i in 1:n
	x_c = -5.0 + 10.0 * rand()
	y_c = -5.0 + 10.0 * rand()
	v_d = 200.0 + (600 - 300) * rand()
	eps = 0.3 + (0.6 - 0.3) * rand()
	pa  = 0.0 + (90.0 - 0.0) * rand()
	push!(comp, (lens=:SIELens, x_c=x_c, y_c, v_d=v_d, x_s=0.5, eps=eps, pa=pa))
end
lens = Lenses.init_CompositeLens(comp)

# Deformation tensor components
dxx, dyy, dxy = Lenses.get_jacobian(lens, x, y)

# Generate singularity map
ra31, ra32, ra41, ra42, rd4 = SingularityMap.from_jacobian(dxx, dyy, dxy, x, y; adis=1.0, D4=true, A4=true)

fig, ax = Lenses.plot_image_plane(lens, x, y, 0.5)
scatter!(ax, ra31[1, :], ra31[2, :], markersize=5, color=:black)
scatter!(ax, ra32[1, :], ra32[2, :], markersize=5, color=:blue)
scatter!(ax, ra41[1, :], ra41[2, :], marker=:star5, markersize=8, color=:cyan)
scatter!(ax, ra42[1, :], ra42[2, :], marker=:star5, markersize=8, color=:cyan)

scatter!(ax, rd4[1, :], rd4[2, :],   marker=:star5, markersize=8, color=:violet)

display(fig)